# DSCI 552, Homework 3

Name: Ivan Sun  
GitHub Username: peterlololsss

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import os
import numpy as np
import pandas as pd

Get the AReM Data Set

In [2]:
columns = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
activities = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]

instances = []
for activity in activities:
    files = [f for f in os.listdir("../data/AReM/" + activity) if f.endswith(".csv")]
    for i in range(1, len(files) + 1):
        path = "../data/AReM/" + activity + "/dataset" + str(i) + ".csv"
        if activity == "bending2" and i == 4:
            series = pd.read_csv(path, skiprows=5, header=None, sep=r"\s+", usecols=range(7))
        else:
            series = pd.read_csv(path, skiprows=5, header=None, usecols=range(7))
        series.columns = columns
        instances.append({"activity": activity, "dataset": i, "series": series})

print("Number of instances:", len(instances))
instances[0]["series"].head()

Number of instances: 88


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


### (b) Test and Train Data

In [3]:
for inst in instances:
    if inst["activity"] in ["bending1", "bending2"]:
        n_test = 2
    else:
        n_test = 3
    if inst["dataset"] <= n_test:
        inst["set"] = "test"
    else:
        inst["set"] = "train"

train_data = [inst for inst in instances if inst["set"] == "train"]
test_data = [inst for inst in instances if inst["set"] == "test"]
print("Train instances:", len(train_data))
print("Test instances:", len(test_data))

Train instances: 69
Test instances: 19


### (c) Feature Extraction

#### i. Research

Time-domain features that are commonly used in time series classification:

- minimum and maximum
- mean and median
- range and interquartile range
- standard deviation and variance
- first and third quartiles
- skewness and kurtosis
- root mean square (RMS) and energy
- mean absolute deviation
- number of peaks and zero crossing rate
- autocorrelation at a given lag
- slope of a linear trend fitted to the series

#### ii. Extraction

In [4]:
rows = []
for inst in instances:
    row = {"activity": inst["activity"], "set": inst["set"]}
    for j in range(6):
        x = inst["series"][columns[j + 1]]
        n = str(j + 1)
        row["min" + n] = x.min()
        row["max" + n] = x.max()
        row["mean" + n] = x.mean()
        row["median" + n] = x.median()
        row["std" + n] = x.std()
        row["1st_quart" + n] = x.quantile(0.25)
        row["3rd_quart" + n] = x.quantile(0.75)
    rows.append(row)

features = pd.DataFrame(rows, index=range(1, len(rows) + 1))
features.index.name = "Instance"
features

,activity,set,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
Instance,,,,,,,,,,,,,,,,,,,,,
1,bending1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,bending1,test,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,bending1,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,bending1,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,bending1,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,walking,train,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,walking,train,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,walking,train,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350


#### iii. Standard Deviation

In [5]:
feature_names = features.columns[2:]
np.random.seed(552)
n_boot = 1000

results = []
for name in feature_names:
    x = features[name].values
    boot_std = []
    for b in range(n_boot):
        sample = np.random.choice(x, size=len(x), replace=True)
        boot_std.append(np.std(sample, ddof=1))
    results.append([name, np.std(x, ddof=1), np.percentile(boot_std, 5), np.percentile(boot_std, 95)])

std_table = pd.DataFrame(results, columns=["feature", "std", "90% CI lower", "90% CI upper"]).set_index("feature")
std_table.round(3)

,std,90% CI lower,90% CI upper
feature,,,
min1,9.570,8.288,10.718
max1,4.394,3.386,5.261
mean1,5.336,4.711,5.897
median1,5.440,4.776,6.045
std1,1.772,1.569,1.957
1st_quart1,6.154,5.606,6.626
3rd_quart1,5.139,4.276,5.833
min2,0.000,0.000,0.000
max2,5.063,4.626,5.389


#### iv. Select Features

I picked mean, standard deviation and max. The mean separates the activities the most: in the var_rss series it is much higher for walking and cycling than for lying, sitting and standing. Median and the two quartiles move together with the mean, so they add little once the mean is in. Standard deviation and max show how much the signal changes over the 2 minutes, which is large for walking and cycling and small for the static activities. Min is a weak choice here, since min2 and min4 are 0 for every instance (std = 0 in iii).

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression should have the lower training RSS. The linear model is the cubic model with $\beta_2 = \beta_3 = 0$, so least squares on the cubic model can always fit the training data at least as well. The extra terms end up fitting some of the noise $\epsilon$.

### (b) Linear Test

The linear regression should have the lower test RSS. The true relationship is linear, so the $X^2$ and $X^3$ terms only fit noise in the training data. This adds variance and makes the cubic test RSS higher.

### (c) Not Linear Train

The cubic regression has the lower training RSS, for the same reason as in (a): it contains the linear model, so its training RSS cannot be higher. Since the true relationship is non-linear, the gap should be even larger here.

### (d) Not Linear Testing

There is not enough information to tell. If the true relationship is close to linear, the linear regression can still have the lower test RSS, since the cubic model adds variance. If it is far from linear, the cubic regression should have the lower test RSS, since its bias is much smaller.

## 3. ISLR 3.7.3 - Extra Practice 

$\hat{Y} = 50 + 20 \, GPA + 0.07 \, IQ + 35 \, Level + 0.01 \, GPA \times IQ - 10 \, GPA \times Level$

(a) iii. For fixed IQ and GPA, college minus high school is $35 - 10 \, GPA$, which is negative once GPA > 3.5. So high school graduates earn more when the GPA is high enough.

(b) $50 + 20(4) + 0.07(110) + 35 + 0.01(4)(110) - 10(4) = 137.1$, so the predicted salary is 137.1 thousand dollars.

(c) False. The size of a coefficient depends on the scale of the variable, and GPA x IQ is in the hundreds. The evidence for an interaction has to come from the p-value (or standard error) of $\hat{\beta}_4$.

## 3. ISLR 3.7.5 - Extra Practice 

$$\hat{y}_i = x_i \hat{\beta} = x_i \frac{\sum_{i'=1}^{n} x_{i'} y_{i'}}{\sum_{j=1}^{n} x_j^2} = \sum_{i'=1}^{n} \frac{x_i x_{i'}}{\sum_{j=1}^{n} x_j^2} \, y_{i'}$$

So $a_{i'} = \dfrac{x_i x_{i'}}{\sum_{j=1}^{n} x_j^2}$.